# Run the full benchmark on a Colab GPU

1. **Runtime > Change runtime type > T4 GPU**, then **Runtime > Run all**.
2. When cell 2 asks, choose the three 10-K PDFs (see `data/README.md`).
3. About 20 to 30 minutes later, `results.zip` downloads.

The generator (Qwen2.5-1.5B-Instruct) and the judge (SmolLM2-1.7B-Instruct) both fit on a T4 in fp16.

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU, then run again."
print(torch.cuda.get_device_name(0))
!git clone -q https://github.com/devpatram/sec-10k-rag-benchmark.git
%cd sec-10k-rag-benchmark
!pip -q install -r requirements.txt

In [ ]:
# The three filings go in data/pdfs. Upload them here (names must match), or fetch them: see data/README.md.
from pathlib import Path
from google.colab import files

need = ["NVIDIA_FY2026_10-K.pdf", "AMD_FY2025_10-K.pdf", "Intel_FY2025_10-K.pdf"]
pdfs = Path("data/pdfs")
pdfs.mkdir(parents=True, exist_ok=True)
missing = [n for n in need if not (pdfs / n).exists()]
if missing:
    print("Choose:", ", ".join(missing))
    for name, data in files.upload().items():
        (pdfs / Path(name).name).write_bytes(data)
still = [n for n in need if not (pdfs / n).exists()]
assert not still, f"Still missing: {still}"
!sha256sum data/pdfs/*.pdf

In [ ]:
!python -m pytest -q
!python -m ragbench.run

In [ ]:
from IPython.display import Image, Markdown, display
display(Markdown(Path("results/summary.md").read_text()))
display(Image("results/comparison.png"))

In [ ]:
!zip -qr results.zip results
files.download("results.zip")